# 7.1. Comm group clean remaining entries to clean manually

Matches comm groups with previously cleaned groups to check if these are already cleaned. Exports remaining uncleaned entries to excel file.

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config

In [ ]:
# Load data
labs = pd.read_csv(config.PROCESSED_DATA / "individual_processed_1.csv", keep_default_na=False, na_values=[""])

share_equip_groups = pd.read_excel(
    config.CLEANING_WORKBOOKS / "groups_cleaning_workbook_final.xlsx", sheet_name="share_equip_groups",
    keep_default_na=False, na_values=[""]
)
share_space_groups = pd.read_excel(
    config.CLEANING_WORKBOOKS / "groups_cleaning_workbook_final.xlsx", sheet_name="share_space_groups",
    keep_default_na=False, na_values=[""]
)

In [3]:
# Reshape labgroupid x entry_i columns to long format
def make_long(prefix, has_comment):
    if has_comment:
        n = max(int(c.split("_")[3]) for c in labs.columns if c.startswith(prefix) and not c.endswith("_co"))
        long_df = pd.concat([
            labs[["labgroupid", f"{prefix}{i}", f"{prefix}{i}_co"]]
            .rename(columns={f"{prefix}{i}": "raw_value", f"{prefix}{i}_co": "comment"})
            for i in range(1, n + 1)
        ], ignore_index=True)
        return long_df.dropna(subset=["raw_value", "comment"], how="all").reset_index(drop=True)
    n = max(int(c.split("_")[-1]) for c in labs.columns if c.startswith(prefix))
    long_df = pd.concat([
        labs[["labgroupid", f"{prefix}{i}"]].rename(columns={f"{prefix}{i}": "raw_value"})
        for i in range(1, n + 1)
    ], ignore_index=True)
    return long_df.dropna(subset=["raw_value"]).reset_index(drop=True)

# Reshape to long for comm groups, equipment sharing groups, and space sharing groups
comm_long = make_long("comm_group_", has_comment=False).drop_duplicates(subset=["labgroupid", "raw_value"])
equip_long = make_long("share_equip_groups_", has_comment=True)
space_long = make_long("share_space_groups_", has_comment=True)

In [4]:
# Attach cleaning status/value from each sheet (their unique key is raw_value + comment)
equip_long = equip_long.merge(
    share_equip_groups[["raw_value", "comment", "cleaned_value", "status"]],
    on=["raw_value", "comment"], how="left", validate="m:1"
)
space_long = space_long.merge(
    share_space_groups[["raw_value", "comment", "cleaned_value", "status"]],
    on=["raw_value", "comment"], how="left", validate="m:1"
)

In [5]:
# Pool of (labgroupid, text, cleaned_value) reported in either sheet, text being either raw_value or comment
cleaned = pd.concat([equip_long, space_long])

pool = pd.concat([
    cleaned[["labgroupid", "raw_value", "cleaned_value"]].rename(columns={"raw_value": "text"}),
    cleaned[["labgroupid", "comment", "cleaned_value"]].rename(columns={"comment": "text"}).dropna(subset=["text"]),
]).drop_duplicates()

In [6]:
# Check for conflicting matches (same text mapping to different, non-missing cleaned_values)
same_lab_conflicts = pool.dropna(subset=["cleaned_value"]).groupby(["labgroupid", "text"])["cleaned_value"].nunique()
any_lab_conflicts = pool.dropna(subset=["cleaned_value"]).groupby("text")["cleaned_value"].nunique()
print(f"{(same_lab_conflicts > 1).sum()} (labgroupid, text) pairs have conflicting cleaned_values")
print(f"{(any_lab_conflicts > 1).sum()} text values have conflicting cleaned_values across labgroupids")

# Collapse to one row per key, joining conflicting cleaned_values (as strings) with " / "; missing values drop out
def join_values(s):
    vals = sorted(set(s.dropna().astype(str)))
    return " / ".join(vals) if vals else np.nan

pool_same_lab = pool.groupby(["labgroupid", "text"])["cleaned_value"].agg(join_values).reset_index()
pool_any_lab = pool.groupby("text")["cleaned_value"].agg(join_values).reset_index()

5 (labgroupid, text) pairs have conflicting cleaned_values
9 text values have conflicting cleaned_values across labgroupids


In [7]:
# Match comm_group entries: same labgroupid first (Certain), then any labgroupid (No labgroupid match)
# A match counts even if cleaned_value is missing - certainty is about whether the text was reported
result = comm_long.merge(
    pool_same_lab.rename(columns={"text": "raw_value", "cleaned_value": "cleaned_certain"}),
    on=["labgroupid", "raw_value"], how="left", indicator="matched_certain"
)
result = result.merge(
    pool_any_lab.rename(columns={"text": "raw_value", "cleaned_value": "cleaned_any_lab"}),
    on="raw_value", how="left", indicator="matched_any_lab"
)

result["cleaned_value"] = result["cleaned_certain"].fillna(result["cleaned_any_lab"])
result["certainty"] = np.select(
    [result["matched_certain"] == "both", result["matched_any_lab"] == "both"],
    ["Certain", "No labgroupid match"],
    default="No match",
)
result["checked"] = np.where(result["certainty"] == "Certain", "Y", "N")

certainty_order = ["Certain", "No labgroupid match", "No match"]
result["certainty"] = pd.Categorical(result["certainty"], categories=certainty_order, ordered=True)
result = result.sort_values("certainty").reset_index(drop=True)

result = result[["labgroupid", "raw_value", "cleaned_value", "certainty", "checked"]]
result["certainty"].value_counts()

certainty
Certain                231
No match               166
No labgroupid match      3
Name: count, dtype: int64

In [8]:
# Export for manual checking
result.to_excel(config.CLEANING_WORKBOOKS / "comm_group_remaining_cleaning.xlsx", index=False)